# detach-clone-snapshot — worked example 2: Reproduce the aliasing bug, then fix it with detach().clone()

> Worked example from [Delta Drills](https://delta-drills.vercel.app). Atom: `detach-clone-snapshot`.

**This is a worked example — read it, run each cell, and follow the reasoning.** It's study material, so there's nothing to submit here. Delta Drills hands you a hands-on version to complete yourself as you get comfortable with the idea.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Concept

_First time on this topic? Run the **Setup** cell above and skim it: every class and helper mentioned below is defined there. You don't need to have done any other drill first._

A classic optimizer-loop bug: appending the live parameter to a history list produces a list of aliases. Since the optimizer updates the parameter's storage in place, every stored reference reflects the *final* value. Snapshotting with `.detach().clone()` gives each entry private storage so the recorded trajectory is correct.

## Worked solution

**Goal:** show both the broken (alias) record and the fixed (snapshot) record side by side.

1. **Set up.** A scalar parameter `w` starts at 1.0; we run a few in-place updates `w -= 0.5` under `no_grad()`.
2. **Broken record.** `bad.append(w)` stores the *same object* each iteration. After the loop, every element of `bad` is the final tensor, so `t.stack(bad)` is constant.
3. **Fixed record.** `good.append(w.detach().clone())` copies the current value into fresh storage each step. `detach()` removes graph tracking (we don't need grads here, but it's the idiomatic snapshot) and `clone()` decouples storage.
4. **Why it works.** In-place `w -= 0.5` writes to `w`'s storage. The `bad` list shares that storage; the `good` list does not, because `clone()` allocated new memory at snapshot time.
5. **Verify.** The `bad` stack is all-equal; the `good` stack is strictly decreasing. Comparing `data_ptr()` confirms the cloned snapshots differ from the live tensor.

In [ ]:
def alias_vs_snapshot(start=1.0, step=0.5, n=4):
    w = t.tensor([start])
    bad, good = [], []
    for _ in range(n):
        with t.no_grad():
            w -= step
        bad.append(w)
        good.append(w.detach().clone())
    return t.stack(bad), t.stack(good), w

bad, good, w = alias_vs_snapshot()
print("bad (aliased) :", bad.squeeze().tolist())
print("good (cloned) :", good.squeeze().tolist())
print("bad all equal final:", bool((bad == w).all()))
print("good shares storage with w:", any(g.data_ptr() == w.data_ptr() for g in good))